# Reflection

我们已经实现过了ReAct以及Plan-And-Solve两种范式，智能体一旦完成了任务，就结束了工作流程。然而他们生成的初始答案和行动轨迹以及最终结果，都可能存在错误以及可改进的空间。

Reflection 机制的核心思想，正是为智能体引入一种 **事后（post-hoc）** 的自我校正循环，使其能够像人类一样，审视自己的工作，发现不足，并进行迭代优化。

## Reflection机制的核心思想

Reflection 机制的灵感来源于人类的学习过程：我们完成初稿后会进行校对，解出数学题后会进行验算。其核心工作流程可以概括为一个简洁的三步循环：**执行 -> 反思 -> 优化**。

1. **执行（Execution）**：智能体首先按照既定的范式（如ReAct或Plan-And-Solve）完成任务，生成初始答案和行动轨迹。
2. **反思（Reflection）**：智能体对自己的输出进行审视，识别其中的错误、不足或改进空间。这一步骤类似于人类在完成任务后进行自我评估的过程。
3. **优化（Optimization）**：基于反思的结果，智能体对初始答案和行动轨迹进行迭代优化，生成更准确、更完善的结果。


与前两种范式相比，Reflection的价值在于：

- 它为智能体提供了一个内部纠错回路，使其不再完全依赖于外部工具的反馈（ReAct 的 Observation），从而能够修正更高层次的逻辑和策略错误。
- 它将一次性的任务执行，转变为一个持续优化的过程，显著提升了复杂任务的最终成功率和答案质量。
- 构建了一个临时的“短期记忆”。整个“执行-反思-优化”的轨迹形成了一个宝贵的经验记录，智能体不仅知道最终答案，还记得自己是如何从有缺陷的初稿迭代到最终版本的。更进一步，这个记忆系统还可以是多模态的，允许智能体反思和修正文本以外的输出（如代码、图像等），为构建更强大的多模态智能体奠定了基础。

## 案例设定与记忆模块设计

为了在实战中体现 Reflection 机制，我们将引入记忆管理机制，因为reflection通常对应着信息的存储和提取，如果上下文足够长的情况，想让“评审员”直接获取所有的信息然后进行反思往往会传入很多冗余信息。这一步实践我们主要完成代码生成与迭代优化。

这一步的目标任务是：“编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。”

这个任务是检验 Reflection 机制的绝佳场景：

1. **存在明确的优化路径**：初始版本的函数可能存在效率低下或逻辑错误，智能体可以通过反思识别这些问题并进行优化。
2. **反思点清晰**：可以通过反思发现其“时间复杂度过高”或“存在重复计算”的问题。
3. **优化方向明确**：智能体可以通过引入更高效的算法（如埃拉托斯特尼筛法）来优化初始版本的函数。

In [1]:
from typing import List, Dict, Any, Optional

class Memory:
    """
    一个简单的短期记忆模块，用于存储智能体的行动与反思轨迹。
    """

    def __init__(self):
        """
        初始化一个空列表来存储所有记录。
        """
        self.records: List[Dict[str, Any]] = []

    def add_record(self, record_type: str, content: str):
        """
        向记忆中添加一条新记录。

        参数:
        - record_type (str): 记录的类型 ('execution' 或 'reflection')。
        - content (str): 记录的具体内容 (例如，生成的代码或反思的反馈)。
        """
        record = {"type": record_type, "content": content}
        self.records.append(record)
        print(f"📝 记忆已更新，新增一条 '{record_type}' 记录。")

    def get_trajectory(self) -> str:
        """
        将所有记忆记录格式化为一个连贯的字符串文本，用于构建提示词。
        """
        trajectory_parts = []
        for record in self.records:
            if record['type'] == 'execution':
                trajectory_parts.append(f"--- 上一轮尝试 (代码) ---\n{record['content']}")
            elif record['type'] == 'reflection':
                trajectory_parts.append(f"--- 评审员反馈 ---\n{record['content']}")

        return "\n\n".join(trajectory_parts)

    def get_last_execution(self) -> Optional[str]:
        """
        获取最近一次的执行结果 (例如，最新生成的代码)。
        如果不存在，则返回 None。
        """
        for record in reversed(self.records):
            if record['type'] == 'execution':
                return record['content']
        return None


这个Memory设计简单，主体是这样的：

- `add_record`方法：用于向记忆中添加新的记录，记录类型可以是“执行”或“反思”，内容则是具体的代码或反馈文本。
- `get_trajectory`方法：将所有的记忆记录格式化为一个连贯的文本字符串，这个字符串可以直接用来构建提示词，帮助智能体在反思阶段回顾之前的尝试和反馈。
- `get_last_execution`方法：提供一个便捷的接口来获取最近一次的执行结果（例如，最新生成的代码），这对于智能体在反思阶段进行针对性的优化非常有用。

## Reflection 智能体的编码实现

有了 Memory 模块作为基础，我们现在可以着手构建 ReflectionAgent 的核心逻辑。整个智能体的工作流程将围绕我们之前讨论的“执行-反思-优化”循环展开，并通过精心设计的提示词来引导大语言模型扮演不同的角色。

### 提示词设计

与之前的范式不同，Reflection 机制需要多个不同角色的提示词来协同工作。

1. **执行者提示词（Executor Prompt）**：引导智能体生成初始代码实现。

In [2]:
INITIAL_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。请根据以下要求，编写一个Python函数。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。

要求: {task}

请直接输出代码，不要包含任何额外的解释。
"""

2.反思提示词：引导智能体扮演一个严格的代码评审员，审视之前生成的代码，识别其中的错误、不足或改进空间，并提供具体的反馈建议。

In [3]:
REFLECT_PROMPT_TEMPLATE = """
你是一位极其严格的代码评审专家和资深算法工程师，对代码的性能有极致的要求。
你的任务是审查以下Python代码，并专注于找出其在<strong>算法效率</strong>上的主要瓶颈。

# 原始任务:
{task}

# 待审查的代码:
```python
{code}
```

请分析该代码的时间复杂度，并思考是否存在一种<strong>算法上更优</strong>的解决方案来显著提升性能。
如果存在，请清晰地指出当前算法的不足，并提出具体的、可行的改进算法建议（例如，使用筛法替代试除法）。
如果代码在算法层面已经达到最优，才能回答“无需改进”。

请直接输出你的反馈，不要包含任何额外的解释。
"""


3.优化提示词：引导智能体根据评审员的反馈，针对性地优化之前生成的代码。

In [4]:

REFINE_PROMPT_TEMPLATE = """
你是一位资深的Python程序员。你正在根据一位代码评审专家的反馈来优化你的代码。

# 原始任务:
{task}

# 你上一轮尝试的代码:
{last_code_attempt}
评审员的反馈：
{feedback}

请根据评审员的反馈，生成一个优化后的新版本代码。
你的代码必须包含完整的函数签名、文档字符串，并遵循PEP 8编码规范。
请直接输出优化后的代码，不要包含任何额外的解释。
"""


## 智能体封装和实现

In [5]:


class ReflectionAgent:
    def __init__(self, llm_client, max_iterations=3):
        self.llm_client = llm_client
        self.memory = Memory()
        self.max_iterations = max_iterations

    def run(self, task: str):
        print(f"\n--- 开始处理任务 ---\n任务: {task}")

        # --- 1. 初始执行 ---
        print("\n--- 正在进行初始尝试 ---")
        initial_prompt = INITIAL_PROMPT_TEMPLATE.format(task=task)
        initial_code = self._get_llm_response(initial_prompt)
        self.memory.add_record("execution", initial_code)

        # --- 2. 迭代循环:反思与优化 ---
        for i in range(self.max_iterations):
            print(f"\n--- 第 {i+1}/{self.max_iterations} 轮迭代 ---")

            # a. 反思
            print("\n-> 正在进行反思...")
            last_code = self.memory.get_last_execution()
            reflect_prompt = REFLECT_PROMPT_TEMPLATE.format(task=task, code=last_code)
            feedback = self._get_llm_response(reflect_prompt)
            self.memory.add_record("reflection", feedback)

            # b. 检查是否需要停止
            if "无需改进" in feedback:
                print("\n✅ 反思认为代码已无需改进，任务完成。")
                break

            # c. 优化
            print("\n-> 正在进行优化...")
            refine_prompt = REFINE_PROMPT_TEMPLATE.format(
                task=task,
                last_code_attempt=last_code,
                feedback=feedback
            )
            refined_code = self._get_llm_response(refine_prompt)
            self.memory.add_record("execution", refined_code)

        final_code = self.memory.get_last_execution()
        print(f"\n--- 任务完成 ---\n最终生成的代码:\n```python\n{final_code}\n```")
        return final_code

    def _get_llm_response(self, prompt: str) -> str:
        """一个辅助方法，用于调用LLM并获取完整的流式响应。"""
        messages = [{"role": "user", "content": prompt}]
        response_text = self.llm_client.think(messages=messages) or ""
        return response_text


In [6]:
from utils.llm import HelloAgentsLLM
llm_client = HelloAgentsLLM()
agent = ReflectionAgent(llm_client=llm_client, max_iterations=3)
task = "编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。"
final_code = agent.run(task=task)


--- 开始处理任务 ---
任务: 编写一个Python函数，找出1到n之间所有的素数 (prime numbers)。

--- 正在进行初始尝试 ---
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
```python
def find_primes(n: int) -> list[int]:
    """
    返回从1到n（包含）之间的所有素数。

    参数:
    n (int): 查找素数的上限（包含）。

    返回:
    list[int]: 包含所有素数的列表，按升序排列。

    异常:
    ValueError: 如果n小于1。

    示例:
    >>> find_primes(10)
    [2, 3, 5, 7]
    >>> find_primes(1)
    []
    """
    if n < 1:
        raise ValueError("n必须大于等于1")

    if n < 2:
        return []

    # 初始化一个布尔列表，表示从2到n的数字是否为素数
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False

    # 使用埃拉托斯特尼筛法
    for i in range(2, int(n ** 0.5) + 1):
        if is_prime[i]:
            # 将i的倍数标记为非素数
            for j in range(i * i, n + 1, i):
                is_prime[j] = False

    # 收集所有素数
    primes = [i for i in range(2, n + 1) if is_prime[i]]
    return primes
```
📝 记忆已更新，新增一条 'execution' 记录。

--- 第 1/3 轮迭代 ---

-> 正在进行反思...
🧠 正在调用 deepseek-chat 模型...
✅ 大语言模型响应成功:
当前代码使用了埃拉托斯特尼筛法，其时间复杂度为 O(n log

这个运行实例展示了 Reflection 机制是如何驱动智能体进行深度优化的:

1.有效的“批判”是优化的前提:在第一轮反思中，由于我们使用了“极其严格”且“专注于算法效率”的提示词，智能体没有满足于功能正确的初版代码，而是精准地指出了其 O(n * sqrt(n)) 的时间复杂度瓶颈，并提出了算法层面的改进建议——埃拉托斯特尼筛法。

## Reflection 机制的成本收益分析

尽管 Reflection 机制在提升任务解决质量上表现出色，但这种能力的获得并非没有代价。在实际应用中，我们需要权衡其带来的收益与相应的成本。

(1)主要成本：
1.模型调用开销：每一次反思和优化都需要额外的模型调用，这可能会显著增加计算资源的消耗和响应时间，尤其是在需要多轮迭代的复杂任务中。

2.任务完成时间：引入反思机制意味着智能体需要经历多个执行-反思-优化的循环，这可能会导致任务完成的总时间增加，特别是在初始版本与最终版本之间存在较大差距的情况下。

3.提示词工程复杂度上升：为了引导智能体进行有效的反思和优化，我们需要设计多个角色的提示词，这增加了提示词工程的复杂度和维护成本。

(2)收益：
1.解决方案质量的跃迁:最大的收益在于，它能将一个“合格”的初始方案，迭代优化成一个“优秀”的最终方案。这种从功能正确到性能高效、从逻辑粗糙到逻辑严谨的提升，在很多关键任务中是至关重要的。

2.鲁棒性和可靠性的提升:通过引入反思机制，智能体能够自我识别和修正错误，从而显著提升了其在面对复杂任务时的鲁棒性和可靠性。

综合来说，Reflection是一种以成本换质量的策略，适用于那些对最终结果质量有较高要求的场景。对于一些对响应时间敏感或资源受限的应用，我们可能需要谨慎评估是否引入反思机制，或者在设计时考虑如何优化反思流程以降低成本。

## 三种范式总结
- **ReAct**：通过引入工具调用的能力，使智能体能够在任务执行过程中动态地获取外部信息，从而提升了其解决复杂任务的能力和灵活性。
- **Plan-And-Solve**：通过将复杂问题分解为一个由多个简单步骤组成的行动计划，使智能体能够更系统地解决问题，提升了其在处理复杂任务时的组织能力和效率。
- **Reflection**：通过引入一个事后的自我校正循环，使智能体能够审视自己的工作，发现不足，并进行迭代优化，从而显著提升了复杂任务的最终成功率和答案质量。